## Creates a new dataframe with processed annotations

### Preparation for pattern db 

### all annotated verb patterns

In [1]:
import pandas as pd
from estnltk import Text

## Configuration

In [2]:
SOURCE_DIR = "../source_data"
# annoteeritud andmed
ANNOTATION_FILE = f"{SOURCE_DIR}/every_verb_case_obl.csv"

## Workflow

### Read in and process verb annotations

In [3]:
margendused = pd.read_csv(ANNOTATION_FILE, sep=";", encoding="utf-8")
margendused

,verbobl,verb,case,isikumäärus,aja-kohamäärus,muu,deprel
0,saama - abl (kellelt/millelt),saama,abl (kellelt/millelt),vahel,vahel,mitte kunagi,obl
1,tulema - abl (kellelt/millelt),tulema,abl (kellelt/millelt),vahel,vahel,mitte kunagi,obl
2,küsima - abl (kellelt/millelt),küsima,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi,obl
3,nõudma - abl (kellelt/millelt),nõudma,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi,obl
4,võtma - abl (kellelt/millelt),võtma,abl (kellelt/millelt),vahel,vahel,mitte kunagi,obl
...,...,...,...,...,...,...,...
10574,musitseerima - in (kelles/milles),musitseerima,in (kelles/milles),mitte kunagi,alati,muu,obl
10575,kõigutama - in (kelles/milles),kõigutama,in (kelles/milles),mitte kunagi,mitte kunagi,muu,obl
10576,kätlema - in (kelles/milles),kätlema,in (kelles/milles),mitte kunagi,alati,muu,obl
10577,kõmmutama - in (kelles/milles),kõmmutama,in (kelles/milles),mitte kunagi,alati,mitte kunagi,obl


#### process_data(annotations)

- võtab sisse:
    - annotastioonide dataframe-i
- vajadusel tekitab ühest kirjest kaks kui verbifraasisi on komaga eraldatud kaks kaassõna
     - nt 'olema kokku, võlgu' -> 'olema kokku' ja 'olema võlgu'
- väljastab:
    - dataframe-i kus ei ole mitme kaassõnaga verbe

In [4]:
def process_data(annotations):
    annotations = annotations.reset_index().drop("index", axis=1)
    
    # check that 'case' contains a space to divide column into case and gov
    # there should be no rows where case doesn't contain a space
    assert len(annotations[annotations["case"].str.contains(" ")==False]) == 0
    
    # split case into phrase case and other
    annotations[["phrase_case", "other"]] = annotations['case'].str.split(' ', n=1, expand=True)
    # split other column into gov and other2
    annotations[["gov", "other2"]] = annotations['other'].str.split(' ', n=1, expand=True)
    # clean gov column
    annotations["gov"]=annotations["gov"].str.replace("(", "").str.replace(")", "")
    # drop unneeded columns 
    annotations = annotations.drop(columns=['other', 'other2'])
    
    df_for_verb_processing = []
    for i in range(len(annotations)):
        row = list(annotations.iloc[i])
        verb = annotations.iloc[i]["verb"].strip()
        if "," in verb:
            osad = verb.split(" ")
            v = osad[0].strip()
            v1 = osad[1].replace(",", "").strip()
            v2 = osad[2].replace(",", "").strip()
            row[1] = v+" "+v1
            new_row1 = row
            df_for_verb_processing.append(row)
            row[1] = v+" "+v2
            new_row2 = row
            df_for_verb_processing.append(row)
        else:
            df_for_verb_processing.append(row)
    df_step1 = pd.DataFrame(df_for_verb_processing, 
                            columns=['verbobl', 'verb', 'case', 'isik', 'koht', 'muu',
                                   'deprel', 'phrase_case', 'gov'])
    # create pattern column
    df_step1['pattern'] = df_step1[['verb', 'gov']].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)
    
    df_step1 = df_step1.drop(columns=['verbobl'])
    df_step1 = df_step1.drop(columns=['case'])
    df_step1 = df_step1.drop(columns=['gov'])
    
    return df_step1

In [5]:
corrected_verbs = process_data(margendused)
corrected_verbs

,verb,isik,koht,muu,deprel,phrase_case,pattern
0,saama,vahel,vahel,mitte kunagi,obl,abl,saama kellelt/millelt
1,tulema,vahel,vahel,mitte kunagi,obl,abl,tulema kellelt/millelt
2,küsima,alati,mitte kunagi,mitte kunagi,obl,abl,küsima kellelt/millelt
3,nõudma,alati,mitte kunagi,mitte kunagi,obl,abl,nõudma kellelt/millelt
4,võtma,vahel,vahel,mitte kunagi,obl,abl,võtma kellelt/millelt
...,...,...,...,...,...,...,...
10601,musitseerima,mitte kunagi,alati,muu,obl,in,musitseerima kelles/milles
10602,kõigutama,mitte kunagi,mitte kunagi,muu,obl,in,kõigutama kelles/milles
10603,kätlema,mitte kunagi,alati,muu,obl,in,kätlema kelles/milles
10604,kõmmutama,mitte kunagi,alati,mitte kunagi,obl,in,kõmmutama kelles/milles


#### process_verbs(df)

- võtab sisse:
    - mustrite/verbide dataframe-i (eelduseks veeru 'verb' olemasolu, mis sisaldab kogu verbifraasi)
- lööb verbifraasi lahku peaverbiks ja komponentideks
- oletus on, et pikema konstruktsiooni viimane verbist liige on peaverb
- eelduslikult on verbis 1 compound
- väljastab: 
    - verbi peasõnade listi
    - verbi komponendi listi

In [6]:
def process_verbs(df):
    verb_word = []
    compound_prt1 = []

    for idx, row in df.iterrows():
        verb = ''
        pieces = row['verb'].strip().split()

        if len(pieces)==1:
            verb = pieces[0]
            compound = ""
        
        else:
            for i in range(len(pieces)):
                text = Text(pieces[i]).tag_layer('morph_analysis')
                if 'V' in text.morph_analysis.partofspeech[0]:
                    verb = pieces[i]
       
            if verb == "" and len(pieces)!=0:
                # if moph analysis did not give a verb, assume first piece is verb
                verb = pieces[0]
                pieces.remove(verb)
                compound = ", ".join(pieces)
            else:
                pieces.remove(verb)
                compound = ", ".join(pieces)

        verb_word.append(verb)
        compound_prt1.append(compound)

    return verb_word, compound_prt1

In [7]:
verbs_list, comp_list1 = process_verbs(corrected_verbs)

In [8]:
print(verbs_list[:10])
print(comp_list1[:10])

['saama', 'tulema', 'küsima', 'nõudma', 'võtma', 'ootama', 'leidma', 'ostma', 'pärinema', 'paluma']
['', '', '', '', '', '', '', '', '', '']


#### create_patterns_dataframe(df, verbs, compunds)

- võtab sisse:
    - dataframe-i, kus on parandatud verbifraasid
    - verbi peasõnade listi
    - verbi compound listi
- lisab tabelisse verbi komponendid ja muu vajaliku info
- väljastab:
    - dataframe-i, kus on vajalik info patterns tabeli jaoks

In [9]:
def create_patterns_dataframe(df1, verb_list, compund_list):
    df1['verb_word'] = verb_list
    df1['verb_compound'] = compund_list
    df1['phrase_nr'] = 1
    df1.insert(0, 'pat_id', range(1, 1 + len(df1)))
    
    df2 = df1.drop(columns=['verb'])
    return df2

In [10]:
df = create_patterns_dataframe(corrected_verbs, verbs_list, comp_list1)
df

,pat_id,isik,koht,muu,deprel,phrase_case,pattern,verb_word,verb_compound,phrase_nr
0,1,vahel,vahel,mitte kunagi,obl,abl,saama kellelt/millelt,saama,,1
1,2,vahel,vahel,mitte kunagi,obl,abl,tulema kellelt/millelt,tulema,,1
2,3,alati,mitte kunagi,mitte kunagi,obl,abl,küsima kellelt/millelt,küsima,,1
3,4,alati,mitte kunagi,mitte kunagi,obl,abl,nõudma kellelt/millelt,nõudma,,1
4,5,vahel,vahel,mitte kunagi,obl,abl,võtma kellelt/millelt,võtma,,1
...,...,...,...,...,...,...,...,...,...,...
10601,10602,mitte kunagi,alati,muu,obl,in,musitseerima kelles/milles,musitseerima,,1
10602,10603,mitte kunagi,mitte kunagi,muu,obl,in,kõigutama kelles/milles,kõigutama,,1
10603,10604,mitte kunagi,alati,muu,obl,in,kätlema kelles/milles,kätlema,,1
10604,10605,mitte kunagi,alati,mitte kunagi,obl,in,kõmmutama kelles/milles,kõmmutama,,1


In [11]:
df.to_csv(f"data/verb_patterns.csv", sep=";", encoding="utf-8", index = False)